In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm import tqdm

# 开启 Pandas 的进度条显示，方便查看计算进度
tqdm.pandas()

def clean_and_process_qm9(input_file="qm9_standard.csv", output_file="qm9_cleaned_final.csv"):
    print(f"正在读取 {input_file} ...")
    
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        print(f"错误：找不到文件 {input_file}。请确保文件在当前目录下。")
        return

    # 1. 检查必要的列是否存在
    # 根据 QM9 的标准命名，列名通常是 'smiles', 'mu', 'gap', 'cv'
    required_cols = ['smiles', 'mu', 'gap', 'cv']
    for col in required_cols:
        if col not in df.columns:
            print(f"错误：CSV 文件中缺少列 '{col}'。请检查列名。")
            print(f"当前列名: {list(df.columns)}")
            return

    # 2. 定义计算 TPSA 的函数
    def calc_tpsa(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                return Descriptors.TPSA(mol)
            return None
        except:
            return None

    # 3. 计算 TPSA (使用 RDKit)
    print("正在使用 RDKit 计算 TPSA (这可能需要几秒钟)...")
    # 使用 progress_apply 显示进度
    df['TPSA'] = df['smiles'].progress_apply(calc_tpsa)

    # 4. 清洗数据
    # 去除 TPSA 计算失败的行（如果有无效 SMILES）
    df_clean = df.dropna(subset=['TPSA'])
    
    # 5. 可选：单位转换 (Hartree -> eV)
    # QM9 的 gap 原始单位是 Hartree，建议转换为 eV 以便符合物理直觉
    # 1 Hartree = 27.2114 eV
    # 如果你需要转换，请取消下面这行的注释：
    # df_clean['gap'] = df_clean['gap'] * 27.2114

    # 6. 只保留指定的列
    final_cols = ['smiles', 'mu', 'gap', 'cv', 'TPSA']
    df_final = df_clean[final_cols]

    # 7. 保存到 CSV
    df_final.to_csv(output_file, index=False)
    
    print(f"\n✅ 处理完成！")
    print(f"原始数据量: {len(df)}")
    print(f"有效数据量: {len(df_final)}")
    print(f"已保存为: {output_file}")
    print("\n前 5 行预览:")
    print(df_final.head())

# 执行主函数
if __name__ == "__main__":
    clean_and_process_qm9()

正在读取 qm9_standard.csv ...
正在使用 RDKit 计算 TPSA (这可能需要几秒钟)...


100%|██████████| 133885/133885 [00:06<00:00, 21213.31it/s]



✅ 处理完成！
原始数据量: 133885
有效数据量: 133885
已保存为: qm9_cleaned_final.csv

前 5 行预览:
  smiles      mu     gap     cv   TPSA
0      C  0.0000  0.5048  6.469   0.00
1      N  1.6256  0.3399  6.316  35.00
2      O  1.8511  0.3615  6.002  31.50
3    C#C  0.0000  0.3351  8.574   0.00
4    C#N  2.8937  0.3796  6.278  23.79
